# VinDatathon 2026 Final Forecasting Notebook

This notebook is the clean judge-facing version of the solution developed in `notebooks/datathon-v3.ipynb`. It is standalone for Kaggle: load the competition files, build deterministic business/calendar features, train the v3 tabular ensemble, and write `submission.csv`.

The core idea is to forecast daily `Revenue` and `COGS` separately on a log scale using Ridge, LightGBM, CatBoost, and quarter-specialist models. The final blend uses the strongest calibration from the v3 experiments.


In [ ]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = Path("/kaggle/input/competitions/datathon-2026-round-1")
if not DATA_DIR.exists():
    DATA_DIR = Path("../dataset") if Path("../dataset").exists() else Path("dataset")

OUTPUT_PATH = Path("submission.csv")

RUN_CONFIG = {
    "weight_scheme": "high_era",
    "q_boost": 3.0,
    "alpha": 0.60,
    "cr": 1.32,
    "cc": 1.36,
}

print("DATA_DIR:", DATA_DIR.resolve())
print(json.dumps(RUN_CONFIG, indent=2))


## Load Data

The model uses the analytical sales target plus promotion, traffic, inventory, and order tables. `customers.csv` is optional and is loaded only for completeness.


In [ ]:

def load_data(data_dir=DATA_DIR):
    sales = pd.read_csv(data_dir / "sales.csv", parse_dates=["Date"])
    sales.columns = ["Date", "Revenue", "COGS"]
    sales = sales.sort_values("Date").reset_index(drop=True)

    test = pd.read_csv(data_dir / "sample_submission.csv", parse_dates=["Date"])
    promotions = pd.read_csv(data_dir / "promotions.csv", parse_dates=["start_date", "end_date"])
    web_traffic = pd.read_csv(data_dir / "web_traffic.csv", parse_dates=["date"])
    inventory = pd.read_csv(data_dir / "inventory.csv", parse_dates=["snapshot_date"])
    orders = pd.read_csv(data_dir / "orders.csv", parse_dates=["order_date"])

    expected = pd.date_range(sales["Date"].min(), sales["Date"].max(), freq="D")
    missing = expected.difference(pd.DatetimeIndex(sales["Date"]))
    if len(missing):
        raise ValueError(f"sales.csv has {len(missing)} missing dates")

    print(f"sales: {sales.shape}, {sales['Date'].min().date()} -> {sales['Date'].max().date()}")
    print(f"test:  {test.shape}, {test['Date'].min().date()} -> {test['Date'].max().date()}")
    return {
        "sales": sales,
        "test": test,
        "promotions": promotions,
        "web_traffic": web_traffic,
        "inventory": inventory,
        "orders": orders,
    }


data = load_data()


## Feature Engineering

The feature layer mirrors `datathon-v3.ipynb`: calendar structure, Fourier seasonality, Tet windows, Vietnam retail holidays, promotion windows, odd/even year interactions, end-of-month effects, and monthly order-volume signal.


In [ ]:

TET_DATES = {
    2012: "2012-01-23", 2013: "2013-02-10", 2014: "2014-01-31",
    2015: "2015-02-19", 2016: "2016-02-08", 2017: "2017-01-28",
    2018: "2018-02-16", 2019: "2019-02-05", 2020: "2020-01-25",
    2021: "2021-02-12", 2022: "2022-02-01", 2023: "2023-01-22",
    2024: "2024-02-10",
}
TET_TS = {year: pd.Timestamp(value) for year, value in TET_DATES.items()}

PROMO_SCHEDULE_DEFAULT = [
    ("spring_sale", 3, 18, 30, 12, True),
    ("mid_year", 6, 23, 29, 18, True),
    ("fall_launch", 8, 30, 32, 10, True),
    ("year_end", 11, 18, 45, 20, True),
    ("urban_blowout", 7, 30, 33, None, "odd"),
    ("rural_special", 1, 30, 30, 15, "odd"),
]

VN_FIXED_HOLIDAYS = [
    (1, 1, "new_year"), (2, 14, "valentine"), (3, 8, "womens_day"),
    (4, 30, "reunification"), (5, 1, "labor_day"), (9, 2, "national_day"),
    (10, 20, "vn_womens_day"), (11, 11, "dd_1111"), (12, 12, "dd_1212"),
    (12, 24, "christmas_eve"), (12, 25, "christmas"),
]


def nearest_tet_diff(dt):
    candidates = []
    for offset in [-1, 0, 1]:
        year = dt.year + offset
        if year in TET_TS:
            diff = (dt - TET_TS[year]).days
            if abs(diff) <= 60:
                candidates.append(diff)
    return min(candidates, key=abs) if candidates else 999


def is_black_friday(dt):
    if dt.month != 11:
        return 0
    last = pd.Timestamp(year=dt.year, month=11, day=30)
    last_friday = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
    return int(dt == last_friday)


def build_business_signals(data):
    wt = data["web_traffic"].copy()
    wt["month"] = wt["date"].dt.month
    if "conversion_rate" not in wt.columns:
        wt["sessions"] = wt["sessions"].replace(0, np.nan)
        wt["conversion_rate"] = 1 - wt["bounce_rate"]

    inv = data["inventory"].copy()
    inv["month"] = inv["snapshot_date"].dt.month

    orders = data["orders"].copy()
    orders["date"] = orders["order_date"].dt.date
    daily_orders = orders.groupby("date").size().reset_index(name="order_count")
    daily_orders["date"] = pd.to_datetime(daily_orders["date"])
    daily_orders["month"] = daily_orders["date"].dt.month
    monthly_order_pattern = daily_orders.groupby("month")["order_count"].mean().reset_index()
    return {"monthly_order_pattern": monthly_order_pattern}


def build_features(dates, business_signals=None, promo_schedule=None):
    if promo_schedule is None:
        promo_schedule = PROMO_SCHEDULE_DEFAULT

    df = pd.DataFrame({"Date": pd.to_datetime(dates)})
    d = df["Date"]

    df["year"] = d.dt.year
    df["month"] = d.dt.month
    df["day"] = d.dt.day
    df["dow"] = d.dt.dayofweek
    df["doy"] = d.dt.dayofyear
    df["quarter"] = d.dt.quarter
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["is_weekday"] = (df["dow"] < 5).astype(int)
    df["days_in_month"] = d.dt.days_in_month
    df["days_to_eom"] = df["days_in_month"] - df["day"]
    df["days_from_som"] = df["day"] - 1
    df["week_of_year"] = d.dt.isocalendar().week.astype(int)

    for k in [1, 2, 3]:
        df[f"is_last{k}"] = (df["days_to_eom"] <= k - 1).astype(int)
        df[f"is_first{k}"] = (df["days_from_som"] <= k - 1).astype(int)
    df["is_last_5d"] = (df["days_to_eom"] <= 4).astype(int)
    df["is_first_5d"] = (df["days_from_som"] <= 4).astype(int)
    df["is_last_10d"] = (df["days_to_eom"] <= 9).astype(int)
    df["is_mid_month"] = ((df["day"] >= 10) & (df["day"] <= 20)).astype(int)

    anchor = pd.Timestamp("2020-01-01")
    df["t_days"] = (d - anchor).dt.days
    df["t_years"] = df["t_days"] / 365.25
    df["regime_pre2019"] = (df["year"] <= 2018).astype(int)
    df["regime_2019"] = (df["year"] == 2019).astype(int)
    df["regime_post2019"] = (df["year"] >= 2020).astype(int)
    df["trend_post2019"] = np.where(df["year"] >= 2020, df["t_days"], 0)
    df["trend_pre2019"] = np.where(df["year"] < 2019, df["t_days"], 0)

    tau = 2 * np.pi
    for k in range(1, 6):
        df[f"sin_y{k}"] = np.sin(tau * k * df["doy"] / 365.25)
        df[f"cos_y{k}"] = np.cos(tau * k * df["doy"] / 365.25)
    for k in range(1, 3):
        df[f"sin_w{k}"] = np.sin(tau * k * df["dow"] / 7.0)
        df[f"cos_w{k}"] = np.cos(tau * k * df["dow"] / 7.0)
        df[f"sin_m{k}"] = np.sin(tau * k * (df["day"] - 1) / df["days_in_month"])
        df[f"cos_m{k}"] = np.cos(tau * k * (df["day"] - 1) / df["days_in_month"])

    diffs = np.array([nearest_tet_diff(dt) for dt in d])
    df["tet_days_diff"] = diffs
    df["tet_in_3"] = (np.abs(diffs) <= 3).astype(int)
    df["tet_in_7"] = (np.abs(diffs) <= 7).astype(int)
    df["tet_in_14"] = (np.abs(diffs) <= 14).astype(int)
    df["tet_before_7"] = ((diffs >= -7) & (diffs < 0)).astype(int)
    df["tet_after_7"] = ((diffs > 0) & (diffs <= 7)).astype(int)
    df["tet_after_14"] = ((diffs > 7) & (diffs <= 21)).astype(int)
    df["tet_on"] = (diffs == 0).astype(int)
    df["tet_proximity"] = np.exp(-0.5 * (diffs / 10) ** 2)

    for month, day, name in VN_FIXED_HOLIDAYS:
        df[f"hol_{name}"] = ((df["month"] == month) & (df["day"] == day)).astype(int)
    df["hol_black_friday"] = [is_black_friday(dt) for dt in d]
    df["hol_double_day"] = (((df["month"] == 11) & (df["day"] == 11)) | ((df["month"] == 12) & (df["day"] == 12))).astype(int)

    df["is_odd_year"] = (df["year"] % 2).astype(int)
    df["is_even_year"] = 1 - df["is_odd_year"]
    for q in range(1, 5):
        df[f"is_q{q}"] = (df["quarter"] == q).astype(int)
    df["q3_odd_year"] = df["is_q3"] * df["is_odd_year"]
    df["q3_even_year"] = df["is_q3"] * df["is_even_year"]

    for name, month in [("jan", 1), ("mar", 3), ("jun", 6), ("aug", 8), ("nov", 11), ("dec", 12)]:
        df[f"{name}_odd"] = ((df["month"] == month) & df["is_odd_year"].astype(bool)).astype(int)
        df[f"{name}_even"] = ((df["month"] == month) & df["is_even_year"].astype(bool)).astype(int)
    for name, month in [("mar", 3), ("jun", 6), ("aug", 8), ("dec", 12)]:
        df[f"eom_{name}"] = df["is_last_5d"] * (df["month"] == month).astype(int)
    for q in [1, 2, 4]:
        df[f"q{q}_odd"] = df[f"is_q{q}"] * df["is_odd_year"]
        df[f"q{q}_even"] = df[f"is_q{q}"] * df["is_even_year"]

    all_years = range(df["year"].min() - 1, df["year"].max() + 2)
    for name, sm, sd, duration, discount, recur in promo_schedule:
        in_promo = np.zeros(len(df), dtype=int)
        since_arr = np.full(len(df), -1.0)
        until_arr = np.full(len(df), -1.0)
        disc_arr = np.zeros(len(df))
        for year in all_years:
            if recur == "odd" and year % 2 == 0:
                continue
            if recur == "even" and year % 2 != 0:
                continue
            start = pd.Timestamp(year=year, month=sm, day=sd)
            end = start + pd.Timedelta(days=duration)
            mask = (d >= start) & (d <= end)
            in_promo[mask] = 1
            since_arr[mask] = (d[mask] - start).dt.days
            until_arr[mask] = (end - d[mask]).dt.days
            disc_arr[mask] = discount or 0
        df[f"promo_{name}"] = in_promo
        df[f"promo_{name}_since"] = since_arr
        df[f"promo_{name}_until"] = until_arr
        df[f"promo_{name}_disc"] = disc_arr

    promo_cols = [c for c in df.columns if c.startswith("promo_") and not any(x in c for x in ["since", "until", "disc"])]
    df["total_active_promos"] = df[promo_cols].sum(axis=1)
    df["has_any_promo"] = (df["total_active_promos"] > 0).astype(int)

    if business_signals is not None:
        order_signal = business_signals["monthly_order_pattern"].set_index("month")
        mean_orders = order_signal["order_count"].mean()
        df["order_volume_signal"] = df["month"].map(order_signal["order_count"] / mean_orders).fillna(1.0)

    for month in range(1, 13):
        df[f"month_{month}"] = (df["month"] == month).astype(int)
    for q in range(1, 5):
        df[f"quarter_{q}"] = (df["quarter"] == q).astype(int)
    for dow in range(7):
        df[f"dow_{dow}"] = (df["dow"] == dow).astype(int)

    df["is_tet_season"] = df["month"].isin([1, 2]).astype(int)
    df["is_spring_peak"] = df["month"].isin([3, 4]).astype(int)
    df["is_summer_low"] = df["month"].isin([7, 8, 9]).astype(int)
    df["is_year_end_peak"] = df["month"].isin([11, 12]).astype(int)
    return df.drop(columns=["Date"])


def compute_sample_weights(dates, scheme="high_era"):
    years = dates.dt.year.values
    if scheme == "high_era":
        weights = np.full(len(dates), 0.01)
        weights[(years >= 2014) & (years <= 2018)] = 1.0
    elif scheme == "blend_recent":
        weights = np.full(len(dates), 0.01)
        weights[(years >= 2014) & (years <= 2018)] = 1.0
        weights[(years >= 2020) & (years <= 2022)] = 0.8
    elif scheme == "post2019_focus":
        weights = np.full(len(dates), 0.01)
        weights[(years >= 2014) & (years <= 2018)] = 0.5
        weights[(years >= 2020) & (years <= 2022)] = 1.0
    else:
        weights = np.ones(len(dates))
    return weights


business_signals = build_business_signals(data)
all_dates = pd.concat([data["sales"]["Date"], data["test"]["Date"]], ignore_index=True)
feat_all = build_features(pd.DatetimeIndex(all_dates), business_signals=business_signals)
feat_all["Date"] = all_dates.values
n_train = len(data["sales"])
feat_train = feat_all.iloc[:n_train].reset_index(drop=True)
feat_test = feat_all.iloc[n_train:].reset_index(drop=True)
feature_cols = [c for c in feat_all.columns if c != "Date"]

print("feature count:", len(feature_cols))
print("missing feature values:", int(feat_all[feature_cols].isna().sum().sum()))


## Models and Blend

All targets are trained as `log(value)`, then exponentiated back to currency scale. Quarter specialists use the same training rows, but boost weights for the target quarter before composing the final horizon prediction by test-date quarter.


In [ ]:

def train_ridge(X_train, y_train, alpha=3.0):
    from sklearn.linear_model import Ridge
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0
    model = Ridge(alpha=alpha, random_state=42, max_iter=5000)
    model.fit((X_train - mu) / sigma, y_train)
    return model, (mu, sigma)


def predict_ridge(model, X_test, stats):
    mu, sigma = stats
    return model.predict((X_test - mu) / sigma)


def train_lgb(X_train, y_train, sample_weight=None, n_rounds_override=None):
    import lightgbm as lgb
    params = dict(
        objective="regression", metric="mae", learning_rate=0.03, num_leaves=63,
        min_data_in_leaf=30, feature_fraction=0.85, bagging_fraction=0.85,
        bagging_freq=5, lambda_l2=1.0, lambda_l1=0.1, seed=42, verbosity=-1, n_jobs=-1,
    )
    if n_rounds_override is None:
        n_val = min(180, int(len(X_train) * 0.1))
        fit_mask = np.ones(len(X_train), dtype=bool)
        fit_mask[-n_val:] = False
        dtrain = lgb.Dataset(X_train[fit_mask], y_train[fit_mask], weight=sample_weight[fit_mask] if sample_weight is not None else None)
        dval = lgb.Dataset(X_train[~fit_mask], y_train[~fit_mask], weight=sample_weight[~fit_mask] if sample_weight is not None else None)
        booster_es = lgb.train(params, dtrain, num_boost_round=5000, valid_sets=[dval], callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(0)])
        best_iter = booster_es.best_iteration
    else:
        best_iter = n_rounds_override
    dtrain_full = lgb.Dataset(X_train, y_train, weight=sample_weight)
    model = lgb.train(params, dtrain_full, num_boost_round=best_iter, callbacks=[lgb.log_evaluation(0)])
    return {"model": model, "best_iter": best_iter}


def train_catboost(X_train, y_train, sample_weight=None):
    from catboost import CatBoostRegressor
    n_val = min(180, int(len(X_train) * 0.1))
    model = CatBoostRegressor(
        iterations=2000, learning_rate=0.03, depth=6, loss_function="MAE",
        eval_metric="MAE", random_seed=42, verbose=0, early_stopping_rounds=300,
    )
    model.fit(
        X_train[:-n_val], y_train[:-n_val],
        sample_weight=sample_weight[:-n_val] if sample_weight is not None else None,
        eval_set=(X_train[-n_val:], y_train[-n_val:]), verbose=False,
    )
    best_iter = max(1, model.best_iteration_ or 2000)
    final_model = CatBoostRegressor(
        iterations=best_iter, learning_rate=0.03, depth=6, loss_function="MAE",
        eval_metric="MAE", random_seed=42, verbose=0,
    )
    final_model.fit(X_train, y_train, sample_weight=sample_weight)
    return final_model


def quarter_specialists(X_train, X_test, y_rev, y_cogs, base_weights, q_train, q_test, q_boost, kind, lgb_best_iters=None):
    pred_rev = np.zeros(len(X_test))
    pred_cogs = np.zeros(len(X_test))
    for quarter in [1, 2, 3, 4]:
        weights = base_weights.copy()
        weights[q_train == quarter] *= q_boost
        if kind == "lgb":
            model_rev = train_lgb(X_train, y_rev, sample_weight=weights, n_rounds_override=lgb_best_iters[0])
            model_cogs = train_lgb(X_train, y_cogs, sample_weight=weights, n_rounds_override=lgb_best_iters[1])
            q_pred_rev = np.exp(model_rev["model"].predict(X_test))
            q_pred_cogs = np.exp(model_cogs["model"].predict(X_test))
        else:
            model_rev = train_catboost(X_train, y_rev, sample_weight=weights)
            model_cogs = train_catboost(X_train, y_cogs, sample_weight=weights)
            q_pred_rev = np.exp(model_rev.predict(X_test))
            q_pred_cogs = np.exp(model_cogs.predict(X_test))
        mask = q_test == quarter
        pred_rev[mask] = q_pred_rev[mask]
        pred_cogs[mask] = q_pred_cogs[mask]
    return pred_rev, pred_cogs


def make_raw_predictions(preds, alpha=0.60):
    lgb_rev = alpha * preds["p_spec_rev"] + (1 - alpha) * preds["p_lgb_rev"]
    lgb_cog = alpha * preds["p_spec_cogs"] + (1 - alpha) * preds["p_lgb_cogs"]
    cat_rev = alpha * preds["p_cat_spec_rev"] + (1 - alpha) * preds["p_cat_rev"]
    cat_cog = alpha * preds["p_cat_spec_cogs"] + (1 - alpha) * preds["p_cat_cogs"]
    raw_rev = 0.10 * preds["p_ridge_rev"] + 0.45 * cat_rev + 0.45 * lgb_rev
    raw_cog = 0.10 * preds["p_ridge_cogs"] + 0.45 * cat_cog + 0.45 * lgb_cog
    return raw_rev, raw_cog


## Train Full Model and Write Submission

This is the final v3 path: train on all historical data, compose the horizon predictions, apply calibration, and save the Kaggle submission file.


In [ ]:

sales = data["sales"]
test = data["test"]

X_train = feat_train[feature_cols].values.astype(np.float32)
X_test = feat_test[feature_cols].values.astype(np.float32)
y_train_rev = np.log(sales["Revenue"].clip(lower=1).values)
y_train_cogs = np.log(sales["COGS"].clip(lower=1).values)
base_weights = compute_sample_weights(sales["Date"], scheme=RUN_CONFIG["weight_scheme"])

print("Training Ridge...")
ridge_rev, stats_rev = train_ridge(X_train, y_train_rev, alpha=3.0)
ridge_cogs, stats_cogs = train_ridge(X_train, y_train_cogs, alpha=3.0)
p_ridge_rev = np.exp(predict_ridge(ridge_rev, X_test, stats_rev))
p_ridge_cogs = np.exp(predict_ridge(ridge_cogs, X_test, stats_cogs))

print("Training LightGBM base...")
lgb_rev = train_lgb(X_train, y_train_rev, sample_weight=base_weights)
lgb_cogs = train_lgb(X_train, y_train_cogs, sample_weight=base_weights)
p_lgb_rev = np.exp(lgb_rev["model"].predict(X_test))
p_lgb_cogs = np.exp(lgb_cogs["model"].predict(X_test))

print("Training CatBoost base...")
cat_rev = train_catboost(X_train, y_train_rev, sample_weight=base_weights)
cat_cogs = train_catboost(X_train, y_train_cogs, sample_weight=base_weights)
p_cat_rev = np.exp(cat_rev.predict(X_test))
p_cat_cogs = np.exp(cat_cogs.predict(X_test))

q_train = feat_train["Date"].apply(lambda value: pd.Timestamp(value).quarter).values
q_test = feat_test["Date"].apply(lambda value: pd.Timestamp(value).quarter).values

print("Training LightGBM quarter specialists...")
p_spec_rev, p_spec_cogs = quarter_specialists(
    X_train, X_test, y_train_rev, y_train_cogs, base_weights, q_train, q_test,
    RUN_CONFIG["q_boost"], kind="lgb", lgb_best_iters=(lgb_rev["best_iter"], lgb_cogs["best_iter"]),
)

print("Training CatBoost quarter specialists...")
p_cat_spec_rev, p_cat_spec_cogs = quarter_specialists(
    X_train, X_test, y_train_rev, y_train_cogs, base_weights, q_train, q_test,
    RUN_CONFIG["q_boost"], kind="catboost",
)

preds = {
    "p_ridge_rev": p_ridge_rev, "p_ridge_cogs": p_ridge_cogs,
    "p_lgb_rev": p_lgb_rev, "p_lgb_cogs": p_lgb_cogs,
    "p_cat_rev": p_cat_rev, "p_cat_cogs": p_cat_cogs,
    "p_spec_rev": p_spec_rev, "p_spec_cogs": p_spec_cogs,
    "p_cat_spec_rev": p_cat_spec_rev, "p_cat_spec_cogs": p_cat_spec_cogs,
}

raw_rev, raw_cog = make_raw_predictions(preds, alpha=RUN_CONFIG["alpha"])
final_rev = np.clip(RUN_CONFIG["cr"] * raw_rev, 1.0, None)
final_cog = np.clip(RUN_CONFIG["cc"] * raw_cog, 1.0, None)

submission = pd.DataFrame({
    "Date": test["Date"].dt.strftime("%Y-%m-%d"),
    "Revenue": np.round(final_rev, 2),
    "COGS": np.round(final_cog, 2),
})
submission.to_csv(OUTPUT_PATH, index=False)

print(f"wrote {OUTPUT_PATH.resolve()} ({len(submission)} rows)")
submission.head()


## Sanity Checks

These checks verify that the generated file matches the expected hidden-test structure and contains positive numeric predictions.


In [ ]:

check = submission.copy()
check["Date"] = pd.to_datetime(check["Date"])
summary = {
    "rows": len(check),
    "date_min": str(check["Date"].min().date()),
    "date_max": str(check["Date"].max().date()),
    "revenue_min": float(check["Revenue"].min()),
    "revenue_mean": float(check["Revenue"].mean()),
    "cogs_min": float(check["COGS"].min()),
    "cogs_mean": float(check["COGS"].mean()),
    "mean_cogs_revenue_ratio": float((check["COGS"] / check["Revenue"]).mean()),
    "null_values": int(check.isna().sum().sum()),
}
print(pd.Series(summary).to_string())

assert list(submission.columns) == ["Date", "Revenue", "COGS"]
assert len(submission) == 548
assert summary["date_min"] == "2023-01-01"
assert summary["date_max"] == "2024-07-01"
assert summary["null_values"] == 0
assert (submission[["Revenue", "COGS"]] > 0).all().all()
